# 7 · Deployment & visualisation

| | |
|---|---|
| Lifecycle stage | 7 of 8: *Deployment & visualisation* (dashboard, model API, user interaction and feedback) |
| Tools | FastAPI, Streamlit, Plotly, Docker, MinIO, Hive Metastore, Trino, ClickHouse, Superset, Grafana |
| Code | `api.py`, `runtime.py`, `online.py`, `scoring.py`, `dashboard/app.py`, `bundle.py`, `publish.py`, `crawler.py`, `olap.py` |

## 7.1 How FraudLens is served

```
                     ┌───────────────────── real time ──────────────────────┐
 card authorisation ─► FastAPI /score ─► online feature store (card history) ─► champion model ─► score, alert, reasons
                     │      ▲                                                                      │
 analyst verdict ─────► /feedback ──► feedback.csv ──► monitoring + next retraining                  ▼
                     │                                                         Prometheus /metrics ─► Grafana
                     └────────────────────────────────────────────────────────────────────────────┘
 batch (daily): pipeline ─► gold Parquet in MinIO ─► Hive Metastore (crawler) ─► Trino ─► Superset dashboards
                                                                       └──────► ClickHouse ─► Grafana business dashboard
 people: Streamlit dashboard (alert queue, live scoring, threshold & cost, patterns, monitoring, model, data)
```

Everything the API and the dashboard need is in the **deploy bundle** (`dashboard/bundle`, about 40 MB):
the `FraudModel` (preprocessor + champion + threshold), the online feature-store snapshot, the scored
transactions and the reports.

In [1]:
import os, sys, json, warnings
from pathlib import Path
ROOT = Path(os.environ.get("FRAUDLENS_ROOT") or (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())).resolve()
CODE = Path(os.environ.get("FRAUDLENS_CODE", ROOT)).resolve()
sys.path.insert(0, str(CODE / "src"))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from fraudlens import viz
from fraudlens.utils import load_config, read_json, read_table

viz.setup()
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"   # small notebooks: plotly.js from the CDN
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
cfg = load_config(root=ROOT)
P = cfg["paths"]
print(f"Project root: {ROOT.name} | data present: {(P['silver'] / 'transactions.parquet').exists()}")
from fraudlens.runtime import example_transaction, load_runtime
bundle = ROOT / "dashboard" / "bundle"
manifest = read_json(bundle / "manifest.json")
print(f"bundle: {len(manifest['files'])} files, {manifest['total_mb']} MB, champion {manifest['champion']}, built {manifest['built_at']}")
print("pinned requirements:", ", ".join(manifest["requirements"]))

Project root: fraudlens | data present: True


bundle: 45 files, 40.3 MB, champion hist_gb, built 2026-10-21T15:25:56
pinned requirements: streamlit>=1.66, pandas>=3.0, numpy==2.4.6, scikit-learn==1.9.1, joblib==1.6.0, pyarrow>=25.0, plotly>=5.24, pyyaml>=6.0, shap>=0.51


## 7.2 Real-time scoring with the online feature store

`load_runtime` loads the bundle exactly as the API does. A transaction arrives with only what the
authorisation system knows (card id, amount, category, time); the feature store derives the card-history
and velocity features from the card's previous transactions, the model scores it, and the alert comes with
reasons an analyst can read.

In [2]:
rt = load_runtime(bundle)
txn = example_transaction(rt.store)
cases = {
    "large online purchase, 30 min after the card's last transaction": txn,
    "same card, small grocery purchase": {**txn, "amount": 12.5, "category": "grocery_pos"},
    "same card, large purchase at 2 am next day": {**txn, "amount": 950.0,
        "ts": (pd.Timestamp(txn["ts"]) + pd.Timedelta(days=1)).replace(hour=2, minute=0).isoformat()},
    "card never seen before (cold start)": {**txn, "card_id": "new-card-0001"},
}
rows = []
for label, t in cases.items():
    r = rt.scorer.score_transaction(t, update_state=False)
    rows.append({"case": label, "amount": r["amount"], "category": r["category"], "score": r["score"], "alert": r["alert"],
                 "known card": r["known_card"], "reasons": "; ".join(r["reasons"][:2])})
pd.DataFrame(rows)

15:30:31 | INFO    | Using a permutation explainer (TreeSHAP values do not add up to the model output (max error 14.163))


15:30:31 | INFO    | Runtime loaded from D:\AUS\CDU\MDS\Semester-3\PRT661\PRT661-A4\fraudlens\dashboard\bundle: hist_gb, threshold 0.7397, 999 cards


,case,amount,category,score,alert,known card,reasons
0,"large online purchase, 30 min after the card's...",875.0000,shopping_net,0.9068,True,True,Amount is 13.0x this card's usual spend; Night...
1,"same card, small grocery purchase",12.5000,grocery_pos,0.0001,False,True,
2,"same card, large purchase at 2 am next day",950.0000,shopping_net,0.4143,False,True,
3,card never seen before (cold start),875.0000,shopping_net,0.9890,True,False,Night-time transaction (23:00); High-risk cate...


In [3]:
import time
t0 = time.perf_counter()
n = 200
for i in range(n):
    rt.scorer.score_transaction({**txn, "amount": 5 + i}, update_state=False)
ms = (time.perf_counter() - t0) / n * 1000
print(f"average in-process scoring latency incl. feature computation and SHAP reasons: {ms:.1f} ms per transaction")

average in-process scoring latency incl. feature computation and SHAP reasons: 161.6 ms per transaction


## 7.3 The model API (FastAPI)

Endpoints: `GET /health`, `GET /model`, `POST /score`, `POST /score/batch`, the review queue `GET /alerts`,
`GET /alerts/{id}`, `POST /alerts/{id}/decision`, `POST /feedback` (verdicts by transaction id),
`GET /monitoring/feedback` and `GET /metrics` (Prometheus). Interactive documentation is served at `/docs`.
Here the app is called in-process with FastAPI's test client, which runs exactly the same code as the
deployed container, with **role-based API keys switched on** (`scorer`, `analyst`).

In [4]:
import importlib, tempfile
os.environ["FRAUDLENS_MODEL_DIR"] = str(bundle)
tmp = Path(tempfile.mkdtemp())
os.environ["FRAUDLENS_FEEDBACK_FILE"] = str(tmp / "feedback.csv")
os.environ["FRAUDLENS_CASE_DB"] = str(tmp / "cases.db")
os.environ["FRAUDLENS_API_KEYS"] = "scorer:demo-scorer-key,analyst:demo-analyst-key"
S, A = {"X-API-Key": "demo-scorer-key"}, {"X-API-Key": "demo-analyst-key"}
from fastapi.testclient import TestClient
import fraudlens.api as api
api = importlib.reload(api)
client = TestClient(api.app)

print("GET /health ->", client.get("/health").json())
print("POST /score without a key ->", client.post("/score", json=txn).status_code, "(401: access control)")
r = client.post("/score", json=txn, params={"update_state": False}, headers=S).json()
print("POST /score ->", json.dumps({k: r[k] for k in ("score", "alert", "threshold", "model", "reasons", "case_id", "routed_to")}, indent=1))
batch = client.post("/score/batch", json=[txn, {**txn, "amount": 8.0}], params={"update_state": False}, headers=S).json()
print("POST /score/batch ->", [(b["amount"], b["score"], b["alert"]) for b in batch])
print("POST /score with amount=-1 ->", client.post("/score", json={**txn, "amount": -1}, headers=S).status_code, "(rejected by validation)")

15:31:22 | INFO    | Using a permutation explainer (TreeSHAP values do not add up to the model output (max error 14.163))


15:31:22 | INFO    | Runtime loaded from D:\AUS\CDU\MDS\Semester-3\PRT661\PRT661-A4\fraudlens\dashboard\bundle: hist_gb, threshold 0.7397, 999 cards


15:31:23 | INFO    | Using a permutation explainer (TreeSHAP values do not add up to the model output (max error 14.163))


15:31:23 | INFO    | Runtime loaded from D:\AUS\CDU\MDS\Semester-3\PRT661\PRT661-A4\fraudlens\dashboard\bundle: hist_gb, threshold 0.7397, 999 cards


GET /health -> {'status': 'ok', 'model': 'hist_gb', 'threshold': 0.739727, 'cards_in_store': 999, 'time': '2026-10-21T15:31:23', 'access_control': 'api-keys'}
POST /score without a key -> 401 (401: access control)


POST /score -> {
 "score": 0.906824,
 "alert": true,
 "threshold": 0.739727,
 "model": "hist_gb",
 "reasons": [
  "Amount is 13.0x this card's usual spend",
  "Night-time transaction (23:00)",
  "6 earlier transactions in 24h totalling $259",
  "Only 30 min since the card's previous transaction",
  "Amount is 7.0 standard deviations above the card's mean",
  "High-risk category (shopping_net: 1.75% fraud rate in training)"
 ],
 "case_id": "C-9b97fe148f",
 "routed_to": "review queue"
}


POST /score/batch -> [(875.0, 0.906824, True), (8.0, 3.2e-05, False)]
POST /score with amount=-1 -> 422 (rejected by validation)


### 7.3.1 A new transaction, end to end: features → inference → review queue → decision → monitoring

A card-testing burst on a known card: six rising amounts three minutes apart, sent to `POST /score` as the
payment system would. Each response shows the features the online store generated, the score against the
threshold and where the transaction was routed. Alerts wait in the review queue; an analyst opens one,
records a decision, and the decision flows into the feedback monitoring.

In [5]:
card = txn["card_id"]
t0 = pd.Timestamp(txn["ts"])
rows = []
for i, amt in enumerate([2.0, 5.0, 25.0, 180.0, 650.0, 1200.0]):
    t = {**txn, "amount": amt, "ts": (t0 + pd.Timedelta(minutes=3 * i)).isoformat(), "transaction_id": f"burst-{i + 1}",
         "category": "shopping_net" if i > 2 else "misc_net", "merchant": "Burst Merchant"}
    out = client.post("/score", json=t, params={"include_features": True}, headers=S).json()
    f = out["features"]
    rows.append({"step": i + 1, "amount": amt, "txns past 1h": f["txn_count_1h"], "spend past 24h": f["amt_sum_24h"],
                 "amount / card mean": round(f["amt_to_card_mean"] or 0, 1), "score": out["score"],
                 "routed to": out["routed_to"], "case": out["case_id"]})
pd.DataFrame(rows)

,step,amount,txns past 1h,spend past 24h,amount / card mean,score,routed to,case
0,1,2.0000,1,258.8300,0.0000,0.0000,approved,C-e6f8cce33a
1,2,5.0000,2,260.8300,0.1000,0.0000,approved,C-d8cef855ab
2,3,25.0000,3,265.8300,0.4000,0.0001,approved,C-405b27d8fa
3,4,180.0000,4,290.8300,2.7000,0.0000,approved,C-8597124ac4
4,5,650.0000,5,470.8300,9.6000,0.2084,approved,C-1ec0e6e8ab
5,6,"1,200.0000",6,"1,120.8300",17.7000,0.9783,review queue,C-a72d56e5e9


In [6]:
queue = client.get("/alerts", headers=A).json()
if not queue:   # no alert in the burst for this card: send one clear night-time outlier
    client.post("/score", headers=S, json={**txn, "amount": 1500.0, "category": "shopping_net", "transaction_id": "night-1",
                                           "ts": (t0.normalize() + pd.Timedelta(days=1, hours=2)).isoformat()})
    queue = client.get("/alerts", headers=A).json()
print(f"review queue: {len(queue)} open case(s)")
case = client.get(f"/alerts/{queue[0]['case_id']}", headers=A).json()
print(json.dumps({k: case[k] for k in ("case_id", "amount", "score", "threshold", "reasons", "clear_below_amount")}, indent=1))
print("top contributions:", [(c["feature"], c["shap"]) for c in case["contributions"][:3]])
print("scorer key on the queue ->", client.get("/alerts", headers=S).status_code, "(403: wrong role)")
d = client.post(f"/alerts/{case['case_id']}/decision", headers=A,
                json={"label": "fraud", "analyst": "notebook-analyst", "note": "customer confirmed card was stolen"}).json()
print("decision ->", d["case"])
fb = client.get("/monitoring/feedback", headers=A).json()
print("feedback monitoring ->", {k: fb[k] for k in ("reviewed_alerts", "confirmed_fraud", "live_precision",
                                                    "live_precision_ci95", "open_alerts", "labelled_examples", "verdict")})
print(api.CASES.audit_log()[["at", "actor", "role", "action", "case_id"]].head(4).to_string(index=False))
metrics = client.get("/metrics").text
print("\n".join(l for l in metrics.splitlines() if l.startswith(("fraudlens_open_alerts", "fraudlens_reviewed_alerts",
                                                                 "fraudlens_live_precision", "fraudlens_access_denied_total"))))
del os.environ["FRAUDLENS_API_KEYS"]

review queue: 3 open case(s)
{
 "case_id": "C-a72d56e5e9",
 "amount": 1200.0,
 "score": 0.978287,
 "threshold": 0.739727,
 "reasons": [
  "Amount is 17.7x this card's usual spend",
  "Night-time transaction (23:00)",
  "11 earlier transactions in 24h totalling $1,121",
  "Only 3 min since the card's previous transaction",
  "Amount is 9.8 standard deviations above the card's mean",
  "High-risk category (shopping_net: 1.75% fraud rate in training)"
 ],
 "clear_below_amount": 281.01
}
top contributions: [('amount', 5.2932), ('amt_sum_1h', 1.8198), ('txn_count_24h', -1.805)]
scorer key on the queue -> 403 (403: wrong role)
decision -> {'case_id': 'C-a72d56e5e9', 'status': 'closed', 'decision': 'fraud', 'analyst': 'notebook-analyst', 'decided_at': '2026-10-21T15:31:28', 'score': 0.978287, 'alert': True}
feedback monitoring -> {'reviewed_alerts': 1, 'confirmed_fraud': 1, 'live_precision': 1.0, 'live_precision_ci95': [0.2065, 1.0], 'open_alerts': 2, 'labelled_examples': 1, 'verdict': 'insuf

fraudlens_open_alerts 2.0
fraudlens_live_precision 1.0
fraudlens_reviewed_alerts 1.0
fraudlens_access_denied_total{endpoint="/score"} 1.0
fraudlens_access_denied_total{endpoint="/alerts"} 1.0


The analyst's decision is a label. With at least 20 decisions, live precision is compared with the 40% policy
floor; `python -m fraudlens.monitor` adds this block to the monitoring summary and exports the decided cases
(with the exact model inputs used) as labelled examples for the next retraining.

## 7.4 The dashboard (Streamlit + Plotly)

`dashboard/app.py` is the analysts' and managers' interface. It runs from the full pipeline output locally,
or from the bundle when hosted (Streamlit Community Cloud or the Docker image). Streamlit's own test harness
renders it headlessly here to prove it starts cleanly from the bundle.

In [7]:
from streamlit.testing.v1 import AppTest
at = AppTest.from_file(str(CODE / "dashboard" / "app.py"), default_timeout=300)
at.run()
print("exceptions:", [e.message for e in at.exception] or "none")
print("tabs:", [t.label for t in at.tabs])
print("headline metrics:", [(m.label, m.value) for m in at.metric][:8])

2026-10-21 15:31:30.943 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


15:31:43 | INFO    | Using a permutation explainer (TreeSHAP values do not add up to the model output (max error 14.163))


15:31:43 | INFO    | Runtime loaded from D:\AUS\CDU\MDS\Semester-3\PRT661\PRT661-A4\fraudlens: hist_gb, threshold 0.7397, 999 cards


exceptions: none
tabs: ['Overview', 'Live transaction', 'Case review', 'Alert history', 'Threshold & cost', 'Patterns', 'Monitoring', 'Model', 'Data pipeline', 'Report']
headline metrics: [('Transactions', '555,719'), ('Alerts raised', '2,851'), ('Cards to review', '531'), ('Frauds in period', '2,145'), ('Precision', '72.4%'), ('Recall', '96.2%'), ('False alarms per catch', '0.4'), ('Estimated net benefit', '$964,182')]


| Tab | Who uses it | What it does |
|---|---|---|
| Overview | manager | KPIs for any period and split, daily alerts and outcomes, money caught and missed, estimated net benefit with its assumptions |
| Live transaction | analyst / demo | a new transaction end to end: received → features → score and explanation → routed to review |
| Case review | analyst | the review queue of live alerts; record decisions; how decisions feed monitoring |
| Alert history | analyst | the replayed test period by card, with SHAP reasons |
| Threshold & cost | manager | move the threshold or the cost assumptions and see recall, precision, alerts/day and estimated net benefit change |
| Patterns | analyst | fraud by category, hour, amount band and state |
| Monitoring | data scientist | monthly drift and performance, monthly stability with intervals, live feedback, retraining recommendation |
| Model | governance | model comparison, selection rule, SHAP importance, explanation quality, transferability to real data |
| Data pipeline | data engineering | lineage from raw to gold, data quality and verification results |
| Report | manager | a plain-language summary of the selected period, downloadable |

## 7.5 Serving the gold layer to BI: lake → catalog → Trino / ClickHouse → Superset / Grafana

`publish.py` writes Hive-partitioned Parquet to MinIO, `crawler.py` registers every table in the Hive
Metastore through Trino (the Glue-crawler role), and `olap.py` loads the gold layer into ClickHouse and
answers the same business questions on both engines. Superset dashboards read Trino; the Grafana business
dashboard reads ClickHouse.

In [8]:
from fraudlens.crawler import VIEWS
from fraudlens.olap import QUERIES
print("Trino views created by the crawler for BI:")
for name, body in VIEWS.items():
    print(f"  hive.{name}: {' '.join(body.split())[:110]}...")
print("\nBusiness questions answered on both Trino and ClickHouse:")
for name, q in QUERIES.items():
    print(f"  {name}: {q['question']}")
for f in ["lake_publication.json", "data_catalog.json", "olap_benchmark.json"]:
    p = P["reports"] / f
    if p.exists():
        r = read_json(p)
        print(f"\n{f}: status={r.get('status')} {r.get('reason', '')}")

Trino views created by the crawler for BI:
  hive.gold.v_alert_queue: SELECT s.transaction_id, s.ts, s.card_id, s.merchant, s.category, s.amount, s.state, s.score, s.outcome, s.rea...
  hive.gold.v_category_performance: SELECT split, category, count(*) AS transactions, sum(is_fraud) AS frauds, sum(alert) AS alerts, sum(CASE WHEN...
  hive.gold.v_state_performance: SELECT split, state, count(*) AS transactions, sum(is_fraud) AS frauds, sum(CASE WHEN outcome = 'caught' THEN ...

Business questions answered on both Trino and ClickHouse:
  daily_outcomes: Fraud caught, missed and false alerts per day (test period)
  category_by_hour: Fraud rate by merchant category and hour of day
  missed_by_merchant: Merchants with the most fraud value the model missed
  alert_precision_by_state: Alert volume and precision by state

lake_publication.json: status=skipped MinIO not reachable at http://localhost:9000

data_catalog.json: status=skipped MinIO not reachable


On a laptop without Docker the platform stages skip themselves (status `skipped` above); the
`platform` workflow runs them for real on a GitHub runner (NiFi → MinIO → Hive Metastore → Trino →
ClickHouse, Superset, Grafana) and publishes the results as a build artefact.

## 7.6 Deployment targets

| Target | How | Command |
|---|---|---|
| Local, no Docker | API + dashboard from the virtual environment | `uvicorn fraudlens.api:app --port 8000` and `streamlit run dashboard/app.py` |
| Container image | `platform/docker/Dockerfile.app`, published to GHCR by `deploy.yml` | `docker run -p 8000:8000 ghcr.io/anasua2905/fraudlens-app api` |
| Full platform | `platform/docker-compose.yml` profiles `lake ingest olap bi mlops` | `docker compose --profile lake --profile mlops up -d` |
| Hosted dashboard | Streamlit Community Cloud from the GitHub repository (`dashboard/app.py`, uses the bundle) | see `docs/deployment.md` |

**Next:** [08 · Monitoring & improvement](08_monitoring_improvement.ipynb)